In [ ]:
# Databricks notebook source


In [ ]:
# Imports
import importlib.util
import json
import logging
import os
import re
import sys
import yaml
from datetime import datetime
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
logger = logging.getLogger(__name__)


In [ ]:
# Runtime inputs
dbutils.widgets.text("days", "1")
dbutils.widgets.text("metadata_catalog", "dev-drugdev_da-koios-catalog")
dbutils.widgets.text("metadata_schema", "dev_drugdev_common")
dbutils.widgets.text("registry_schema", "dev_drugdev_common")
dbutils.widgets.text("environment", "dev")
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("bronze_schema", "")
dbutils.widgets.text("silver_schema", "")

try:
    days = int((dbutils.widgets.get("days") or "1").strip())
except Exception:
    days = 1
days = max(days, 1)

metadata_catalog = (dbutils.widgets.get("metadata_catalog") or "").strip()
metadata_schema = (dbutils.widgets.get("metadata_schema") or "").strip()
registry_schema = (dbutils.widgets.get("registry_schema") or "").strip()
environment_widget = (dbutils.widgets.get("environment") or "").strip() or "dev"
catalog_widget = (dbutils.widgets.get("catalog") or "").strip()
bronze_schema_widget = (dbutils.widgets.get("bronze_schema") or "").strip()
silver_schema_widget = (dbutils.widgets.get("silver_schema") or "").strip()

if metadata_catalog:
    spark.conf.set("drugdev.METADATA_CATALOG", metadata_catalog)
if metadata_schema:
    spark.conf.set("drugdev.METADATA_SCHEMA", metadata_schema)
if registry_schema:
    spark.conf.set("drugdev.REGISTRY_SCHEMA", registry_schema)
if environment_widget:
    spark.conf.set("drugdev.environment", environment_widget)
if catalog_widget:
    spark.conf.set("drugdev.CATALOG", catalog_widget)
if bronze_schema_widget:
    spark.conf.set("drugdev.BRONZE_SCHEMA", bronze_schema_widget)
if silver_schema_widget:
    spark.conf.set("drugdev.SILVER_SCHEMA", silver_schema_widget)

print(f"Pipeline summary window: last {days} day(s)")
print(f"Configured metadata: catalog={spark.conf.get('drugdev.METADATA_CATALOG', '')} schema={spark.conf.get('drugdev.METADATA_SCHEMA', '')}")

In [ ]:
# Resolve required Spark conf values
required_keys = [
    "drugdev.METADATA_CATALOG",
    "drugdev.METADATA_SCHEMA",
]
conf = {}
missing = []
for key in required_keys:
    value = spark.conf.get(key, "").strip()
    if not value:
        missing.append(key)
    conf[key.split(".")[-1]] = value

if missing:
    raise ValueError(f"Missing required Spark conf(s): {', '.join(missing)}")

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
notebook_path = ctx.notebookPath().get()
NOTEBOOK_DIR = f"/Workspace{notebook_path.rsplit('/', 1)[0]}"
WORKSPACE_ROOT = NOTEBOOK_DIR.rsplit('/databricks_bundle/src', 1)[0]
FILES_ROOT = WORKSPACE_ROOT if WORKSPACE_ROOT.endswith('/files') else f"{WORKSPACE_ROOT}/files"
NOTIFICATION_DIR = os.path.abspath(f"{FILES_ROOT}/databricks_bundle/drugdev/notification_faramework")
environment = spark.conf.get("drugdev.environment", "dev").strip() or "dev"

metrics_tbl = f"`{conf['METADATA_CATALOG']}`.{conf['METADATA_SCHEMA']}.pipeline_execution_metrics"
runtime_tbl = f"`{conf['METADATA_CATALOG']}`.{conf['METADATA_SCHEMA']}.ingestion_runtime_state"
dq_tbl = f"`{conf['METADATA_CATALOG']}`.{conf['METADATA_SCHEMA']}.dq_validation_results_log"
registry_schema = spark.conf.get("drugdev.REGISTRY_SCHEMA", "").strip()
dataset_registry_tbl = (
    f"`{conf['METADATA_CATALOG']}`.{registry_schema}.dataset_registry"
    if registry_schema
    else ""
)

print(f"Metrics table: {metrics_tbl}")
print(f"Bronze runtime table: {runtime_tbl}")
print(f"DQ results table: {dq_tbl}")


def import_from_path(name, path):
    module_dir = os.path.dirname(os.path.abspath(path))
    if module_dir not in sys.path:
        sys.path.insert(0, module_dir)
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    mod.dbutils = dbutils
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod


def _as_bool(v: str) -> bool:
    return str(v).strip().lower() in {"1", "true", "yes", "y", "on"}


def _get_conf_required(key: str) -> str:
    value = spark.conf.get(key, "").strip()
    if not value:
        raise ValueError(f"Missing required Spark conf: {key}")
    return value


def _get_conf_optional(key: str, default: str = "") -> str:
    return spark.conf.get(key, default).strip()


def _json_list_from_conf(key: str, required: bool = False) -> list:
    raw = _get_conf_required(key) if required else _get_conf_optional(key, "[]")
    try:
        parsed = json.loads(raw) if raw else []
    except Exception as exc:
        raise ValueError(f"Spark conf {key} must be valid JSON list") from exc
    if not isinstance(parsed, list):
        raise ValueError(f"Spark conf {key} must be a JSON list")
    return parsed


def _json_dict_from_conf(key: str, default: str = "{}") -> dict:
    raw = _get_conf_optional(key, default)
    try:
        parsed = json.loads(raw) if raw else {}
    except Exception as exc:
        raise ValueError(f"Spark conf {key} must be valid JSON object") from exc
    if not isinstance(parsed, dict):
        raise ValueError(f"Spark conf {key} must be a JSON object")
    return parsed


def _build_alerting_config_from_spark() -> dict:
    email_enabled = _as_bool(_get_conf_optional("drugdev.NOTIFY_EMAIL_ENABLED", "false"))
    teams_enabled = _as_bool(_get_conf_optional("drugdev.NOTIFY_TEAMS_ENABLED", "false"))

    missing = []
    if email_enabled:
        for k in [
            "drugdev.NOTIFY_EMAIL_FROM",
            "drugdev.NOTIFY_EMAIL_REPLY_TO",
            "drugdev.NOTIFY_EMAIL_RECIPIENTS_CRITICAL_JSON",
            "drugdev.NOTIFY_EMAIL_RECIPIENTS_HIGH_JSON",
            "drugdev.NOTIFY_EMAIL_RECIPIENTS_MEDIUM_JSON",
        ]:
            if not spark.conf.get(k, "").strip():
                missing.append(k)
    if teams_enabled and not spark.conf.get("drugdev.NOTIFY_TEAMS_WEBHOOK_URL", "").strip():
        missing.append("drugdev.NOTIFY_TEAMS_WEBHOOK_URL")
    if missing:
        raise ValueError("Missing required Spark conf(s) for alerting: " + ", ".join(missing))

    email_cfg = {
        "enabled": email_enabled,
        "smtp_driver": _get_conf_optional("drugdev.NOTIFY_EMAIL_DRIVER", "ses"),
        "from_address": _get_conf_optional("drugdev.NOTIFY_EMAIL_FROM"),
        "reply_to": _get_conf_optional("drugdev.NOTIFY_EMAIL_REPLY_TO"),
        "ses_region": _get_conf_optional("drugdev.NOTIFY_EMAIL_SES_REGION", _get_conf_optional("AWS_DEFAULT_REGION", "us-west-2")),
        "recipients": {
            "critical": _json_list_from_conf("drugdev.NOTIFY_EMAIL_RECIPIENTS_CRITICAL_JSON") if email_enabled else [],
            "high": _json_list_from_conf("drugdev.NOTIFY_EMAIL_RECIPIENTS_HIGH_JSON") if email_enabled else [],
            "medium": _json_list_from_conf("drugdev.NOTIFY_EMAIL_RECIPIENTS_MEDIUM_JSON") if email_enabled else [],
            "low": _json_list_from_conf("drugdev.NOTIFY_EMAIL_RECIPIENTS_LOW_JSON"),
        },
    }

    teams_cfg = {
        "enabled": teams_enabled,
        "webhook_url": _get_conf_optional("drugdev.NOTIFY_TEAMS_WEBHOOK_URL"),
    }

    routing_cfg = _json_dict_from_conf(
        "drugdev.NOTIFY_ROUTING_JSON",
        '{"CRITICAL":{"teams":true},"HIGH":{"teams":true},"MEDIUM":{"teams":false},"LOW":{"teams":false}}',
    )
    domain_recipients_cfg = _json_dict_from_conf("drugdev.NOTIFY_DOMAIN_RECIPIENTS_JSON", "{}")

    return {
        "notifications": {
            "email": email_cfg,
            "teams": teams_cfg,
            "routing": routing_cfg,
            "domain_recipients": domain_recipients_cfg,
        }
    }


if not spark.conf.get("drugdev.CATALOG", "").strip():
    spark.conf.set("drugdev.CATALOG", conf["METADATA_CATALOG"])
if not spark.conf.get("drugdev.BRONZE_SCHEMA", "").strip():
    spark.conf.set("drugdev.BRONZE_SCHEMA", f"{environment}_drugdev_bronze")
if not spark.conf.get("drugdev.SILVER_SCHEMA", "").strip():
    spark.conf.set("drugdev.SILVER_SCHEMA", f"{environment}_drugdev_silver")
if not spark.conf.get("drugdev.REGISTRY_SCHEMA", "").strip():
    spark.conf.set("drugdev.REGISTRY_SCHEMA", registry_schema or f"{environment}_drugdev_common")
if not spark.conf.get("drugdev.environment", "").strip():
    spark.conf.set("drugdev.environment", environment)
if not spark.conf.get("drugdev.CONFIG_PATH", "").strip():
    default_config_path = os.path.abspath(f"{NOTEBOOK_DIR}/../../config/environments/{environment}.yaml")
    spark.conf.set("drugdev.CONFIG_PATH", default_config_path)

alert_notifier = import_from_path(
    "alert_notifier",
    f"{NOTIFICATION_DIR}/alert_notifier.py"
)
alert_cfg = _build_alerting_config_from_spark()
notifier = alert_notifier.AlertNotifier(alert_cfg)


In [ ]:
# Execution detail
exec_df = spark.sql(f"""
    SELECT
        job_name,
        task_name,
        domain,
        layer,
        status,
        duration_seconds,
        records_written,
        start_time,
        end_time,
        error_message
    FROM {metrics_tbl}
    WHERE execution_timestamp >= CURRENT_TIMESTAMP() - INTERVAL {days} DAYS
    ORDER BY start_time DESC
""")
print("=== Recent task executions ===")
display(exec_df)


In [ ]:
# Bronze dataset runtime detail
if dataset_registry_tbl:
    bronze_df = spark.sql(f"""
        SELECT
            dr.dataset_name,
            irs.dataset_id,
            UPPER(COALESCE(irs.last_run_status, 'UNKNOWN')) AS status,
            irs.records_ingested,
            irs.files_processed,
            irs.last_run_start_time,
            irs.last_run_end_time,
            irs.failure_reason
        FROM {runtime_tbl} irs
        LEFT JOIN {dataset_registry_tbl} dr
          ON dr.dataset_id = irs.dataset_id
        WHERE COALESCE(irs.last_run_end_time, irs.last_run_start_time) >= CURRENT_TIMESTAMP() - INTERVAL {days} DAYS
        ORDER BY COALESCE(irs.last_run_end_time, irs.last_run_start_time) DESC
    """)
else:
    bronze_df = spark.sql(f"""
        SELECT
            dataset_id,
            UPPER(COALESCE(last_run_status, 'UNKNOWN')) AS status,
            records_ingested,
            files_processed,
            last_run_start_time,
            last_run_end_time,
            failure_reason
        FROM {runtime_tbl}
        WHERE COALESCE(last_run_end_time, last_run_start_time) >= CURRENT_TIMESTAMP() - INTERVAL {days} DAYS
        ORDER BY COALESCE(last_run_end_time, last_run_start_time) DESC
    """)

print("=== Bronze dataset runtime detail ===")
display(bronze_df)


In [ ]:
# Silver DQ rule-level summary
dq_df = spark.sql(f"""
    SELECT
        domain,
        vendor,
        study_id,
        table_name,
        COUNT(*) AS rules_evaluated,
        SUM(CASE WHEN UPPER(result) = 'SUCCESS' THEN 1 ELSE 0 END) AS rules_passed,
        SUM(CASE WHEN UPPER(result) <> 'SUCCESS' THEN 1 ELSE 0 END) AS rules_failed,
        ROUND(AVG(pass_rate_pct), 2) AS avg_pass_rate_pct,
        MAX(validation_timestamp) AS last_validation_ts
    FROM {dq_tbl}
    WHERE validation_timestamp >= CURRENT_TIMESTAMP() - INTERVAL {days} DAYS
    GROUP BY domain, vendor, study_id, table_name
    ORDER BY rules_failed DESC, domain, vendor, study_id, table_name
""")

print("=== Silver DQ summary ===")
display(dq_df)


In [ ]:
# Layer summary
summary_df = spark.sql(f"""
    SELECT
        layer,
        COUNT(*) AS task_runs,
        SUM(CASE WHEN status = 'SUCCESS' THEN 1 ELSE 0 END) AS success_runs,
        SUM(CASE WHEN status = 'FAILED' THEN 1 ELSE 0 END) AS failed_runs,
        COALESCE(SUM(records_written), 0) AS total_records_written,
        COALESCE(AVG(duration_seconds), 0) AS avg_duration_seconds
    FROM {metrics_tbl}
    WHERE execution_timestamp >= CURRENT_TIMESTAMP() - INTERVAL {days} DAYS
    GROUP BY layer
    ORDER BY layer
""")
print("=== Layer summary ===")
display(summary_df)


In [ ]:
# Gold object execution detail
gold_df = spark.sql(f"""
        SELECT
                run_id,
                task_name AS object_name,
                status,
                duration_seconds,
                records_written,
                start_time,
                end_time,
                error_message
        FROM {metrics_tbl}
        WHERE execution_timestamp >= CURRENT_TIMESTAMP() - INTERVAL {days} DAYS
            AND UPPER(layer) = 'GOLD'
        ORDER BY start_time DESC
""")
print("=== Gold object execution detail ===")
display(gold_df)


In [ ]:
# Final status
failed_count = spark.sql(f"""
    SELECT COUNT(*) AS c
    FROM {metrics_tbl}
    WHERE execution_timestamp >= CURRENT_TIMESTAMP() - INTERVAL {days} DAYS
      AND status = 'FAILED'
""").first()['c']

bronze_failed_count = spark.sql(f"""
        SELECT COUNT(*) AS c
        FROM {runtime_tbl}
        WHERE COALESCE(last_run_end_time, last_run_start_time) >= CURRENT_TIMESTAMP() - INTERVAL {days} DAYS
            AND LOWER(COALESCE(last_run_status, '')) = 'failed'
""").first()['c']

dq_failed_count = spark.sql(f"""
        SELECT COUNT(*) AS c
        FROM {dq_tbl}
        WHERE validation_timestamp >= CURRENT_TIMESTAMP() - INTERVAL {days} DAYS
            AND UPPER(COALESCE(result, '')) <> 'SUCCESS'
""").first()['c']

overall_failed = int(failed_count) + int(bronze_failed_count) + int(dq_failed_count)
overall_status = 'FAILED' if overall_failed > 0 else 'SUCCESS'

summary_text = (
    f"Workflow summary [{environment}] status={overall_status}; "
    f"window_days={days}; pipeline_failed_tasks={failed_count}; "
    f"bronze_failed_datasets={bronze_failed_count}; dq_failed_rules={dq_failed_count}"
)
print(summary_text)

try:
    notifier.send_alert(
        alert_type='WORKFLOW_SUMMARY',
        severity='CRITICAL' if overall_status == 'FAILED' else 'MEDIUM',
        title=f"Workflow Summary [{environment}] - {overall_status}",
        message=summary_text[:400],
        context={
            'status': overall_status,
            'days': days,
            'pipeline_failed_tasks': int(failed_count),
            'bronze_failed_datasets': int(bronze_failed_count),
            'dq_failed_rules': int(dq_failed_count),
            'generated_at_utc': datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S'),
        },
    )
except Exception as alert_exc:
    logger.warning("Failed to send workflow summary alert: %s", alert_exc)

dbutils.notebook.exit(overall_status)
